In [1]:
import optuna
import axelrod as axl
from axelrod.action import Action, actions_to_str
from axelrod.player import Player
from axelrod.strategy_transformers import (
    FinalTransformer,
    TrackHistoryTransformer,
)
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import numpy as np
from collections import Counter
import pandas as pd
from math import exp
import numpy as np


c:\Users\Ognjen\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
C, D = Action.C, Action.D

class FuzzyMethods():
    @staticmethod
    def sigmoid(x, center, scale=10):
        return 1 / (1 + exp(-scale * (x - center)))
    
    @staticmethod
    def fuzzy_gate(mu_D, d_thresh, mu_C, c_thresh):
        d_condition = FuzzyMethods.sigmoid(mu_D, center=d_thresh)
        c_condition = 1 - FuzzyMethods.sigmoid(mu_C, center=c_thresh)
  
        w1 = d_condition * c_condition
          
        return w1
    
    @staticmethod
    def calc_cooperation(self, opponent):
        return (Counter(opponent.history)[C])/len(opponent.history)*100
    
    @staticmethod
    def calc_adaptivity(self, opponent):
        adapCounter = 0
        adapReaction = 0

        if(len(self.history) < 3): 
            return 0
        
        for i in range(3, len(self.history)):
            if (self.history[i-3] == C and self.history[i-2] == D):
                adapCounter += 1
                if (opponent.history[i-1] == D):
                    adapReaction += 1
                elif (self.history[i-1] == D and opponent.history[i] == D):
                    adapReaction += 0.5
            elif (self.history[i-3] == D and self.history[i-2] == C):
                adapCounter += 1
                if (opponent.history[i-1] == C):
                    adapReaction += 1
                elif (self.history[i-1] == C and opponent.history[i] == C):
                    adapReaction += 0.5

        if adapCounter == 0:
            return 0
        
        return adapReaction/adapCounter*100

    
    @staticmethod
    def calc_forgiveness(self, opponent):
        DCounter = 0
        punishmentCounter = 0

        for i in range(0, len(self.history)-1):
            if(self.history[i] == D):
                DCounter += 1
                for j in range (i+1, len(opponent.history)):
                    if(opponent.history[j] == C):
                        break
                    else:
                        punishmentCounter += 1
              

        if punishmentCounter > 0:
            return DCounter/punishmentCounter*100
        else:
            return 100
    
    @staticmethod
    def calc_stochastic(self, opponent):
        patterns = [
            [C, C, C],
            [C, C, D],
            [C, D, C],
            [C, D, D],
            [D, C, C],
            [D, C, D],
            [D, D, C],
            [D, D, D]
        ]

        non_stochasticCounter = 0
        patternPlayedCounter = 0

        for p in patterns:
            opponentsReactions = []
            for i in range(0, len(self.history)-3):
                if ([self.history[i], self.history[i+1], self.history[i+2]] == p):
                    opponentsReactions.append([opponent.history[i+1], opponent.history[i+2], opponent.history[i+3]])
            
            unique_patterns = len(set(tuple(sub) for sub in opponentsReactions))

            if(len(opponentsReactions) > 0):
                non_stochasticCounter += 0 if unique_patterns == 1 else unique_patterns
                patternPlayedCounter += len(opponentsReactions)
        
        if patternPlayedCounter == 0:
            return 0
        
        return non_stochasticCounter/patternPlayedCounter*100

In [3]:
def build_player(params): 
    class FSOptimized(Player):

        name = "FuzzySugeno"
        classifier = {
            "memory_depth": float("inf"),
            "stochastic": False,
            "long_run_time": False,
            "inspects_source": False,
            "manipulates_source": False,
            "manipulates_state": False,
        }

        cooperation = ctrl.Antecedent(np.arange(0, 100, 1), 'cooperation')
        adaptivity = ctrl.Antecedent(np.arange(0, 100, 1), 'adaptivity')
        forgiveness = ctrl.Antecedent(np.arange(0, 100, 1), 'forgiveness')
        stochastic = ctrl.Antecedent(np.arange(0, 100, 1), 'stochastic')

        cooperation.automf(names=["low", "medium", "high"])
        adaptivity.automf(names=["no", "yes"])
        forgiveness.automf(names=["low", "medium", "high"])
        forgiveness['low'] = fuzz.gaussmf(forgiveness.universe, 0, 25)
        forgiveness['medium'] = fuzz.trimf(forgiveness.universe, [25, 50, 75])
        stochastic.automf(names=["none", "sometimes", "always"])

        resulting_strategy = ctrl.Consequent(np.arange(0, 100, 1),'resulting_strategy')
        resulting_strategy['D'] = fuzz.trimf(resulting_strategy.universe, [params['d_threshold_1'], params['d_threshold_2'], params['d_threshold_3']])
        resulting_strategy['C'] = fuzz.trimf(resulting_strategy.universe, [params['c_threshold_1'], params['c_threshold_2'], params['c_threshold_3']])

        rule1 = ctrl.Rule(cooperation['high'] & adaptivity['no'] & (forgiveness['medium'] | forgiveness['high']), resulting_strategy['D'])
        rule2 = ctrl.Rule(forgiveness['low'] & cooperation['high'], resulting_strategy['C'])
        rule3 = ctrl.Rule(stochastic['always'] | adaptivity['no'], resulting_strategy['D'])
        rule4 = ctrl.Rule(cooperation['low'] | (cooperation['medium'] & forgiveness['low']), resulting_strategy['D'])
        rule5 = ctrl.Rule(cooperation['medium'] & forgiveness['medium'] & adaptivity['yes'], resulting_strategy['C'])

        strategy_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5])
        chosen_strategy = ctrl.ControlSystemSimulation(strategy_ctrl)

        h = {'Name':'', 'Fuzzy' : [], 'Opponent': []}

        forgiveness_index = 0
        stochastic_index = 0
        first_time = True
    
        def strategy(self, opponent: Player) -> Action:

            if(len(self.history) == 0 or D not in opponent.history):
                return C
            
            else:
                coop = FuzzyMethods.calc_cooperation(self, opponent)
                adap = FuzzyMethods.calc_adaptivity(self, opponent)
                forg = FuzzyMethods.calc_forgiveness(self, opponent)
                stoch = FuzzyMethods.calc_stochastic(self, opponent)

                self.chosen_strategy.input['cooperation'] = coop
                self.chosen_strategy.input['adaptivity'] = adap
                self.chosen_strategy.input['forgiveness'] = forg
                self.chosen_strategy.input['stochastic'] = stoch
                
                self.chosen_strategy.compute()

                d_membership = fuzz.interp_membership(self.resulting_strategy.universe, self.resulting_strategy['D'].mf, self.chosen_strategy.output['resulting_strategy'])
                c_membership = fuzz.interp_membership(self.resulting_strategy.universe, self.resulting_strategy['C'].mf, self.chosen_strategy.output['resulting_strategy'])

                output = FuzzyMethods.fuzzy_gate(d_membership, 0.5, c_membership, 0.5)

                if(output > 0.5):
                    return D
            
            return C
    return FSOptimized()

In [4]:
def fitness_function(solution, solution_idx):
    print(f"Evaluating solution {solution_idx}: {solution}")

    results = axl.Tournament(players=[build_player(solution)] + [player() for player in axl.stewart_plotkin_strategies], turns=200).play(progress_bar=False)
    return np.mean(results.normalised_scores[0])


In [5]:
j = 0
def objective(trial):
    global j

    d_threshold_1 = trial.suggest_int('d_threshold_1', 0, 99)
    d_threshold_2 = trial.suggest_int('d_threshold_2', d_threshold_1, 99)
    d_threshold_3 = trial.suggest_int('d_threshold_3', d_threshold_2, 99)

    c_threshold_1 = trial.suggest_int('c_threshold_1', 0, 99)
    c_threshold_2 = trial.suggest_int('c_threshold_2', c_threshold_1, 99)
    c_threshold_3 = trial.suggest_int('c_threshold_3', c_threshold_2, 99)

    params = {'d_threshold_1': d_threshold_1, 'd_threshold_2': d_threshold_2, 'd_threshold_3': d_threshold_3, 'c_threshold_1': c_threshold_1, 'c_threshold_2': c_threshold_2, 'c_threshold_3': c_threshold_3}
    try:
        return fitness_function(params, 0)

    except Exception as e:
        print(f"Failed to build player: {e}")
        return 0.0

In [6]:
study = optuna.create_study(
    direction='maximize',
    study_name='fuzzy_optimization_sugeno_3',
    storage='sqlite:///fuzzy_optimization_sugeno_3.db',
    load_if_exists=True
)
study.enqueue_trial({
        'd_threshold_1': 0,
        'd_threshold_2': 25,
        'd_threshold_3': 50,
        'c_threshold_1': 35,
        'c_threshold_2': 75,
        'c_threshold_3': 99,

})
study.optimize(objective, n_trials=300, show_progress_bar=True)

[I 2026-11-21 08:41:19,502] A new study created in RDB with name: fuzzy_optimization_sugeno_3
  0%|          | 0/300 [00:00<?, ?it/s]

Evaluating solution 0: {'d_threshold_1': 0, 'd_threshold_2': 25, 'd_threshold_3': 50, 'c_threshold_1': 35, 'c_threshold_2': 75, 'c_threshold_3': 99}


Best trial: 0. Best value: 2.70396:   0%|          | 1/300 [00:34<2:54:07, 34.94s/it]

[I 2026-11-21 08:41:54,461] Trial 0 finished with value: 2.703964285714286 and parameters: {'d_threshold_1': 0, 'd_threshold_2': 25, 'd_threshold_3': 50, 'c_threshold_1': 35, 'c_threshold_2': 75, 'c_threshold_3': 99}. Best is trial 0 with value: 2.703964285714286.
Evaluating solution 0: {'d_threshold_1': 43, 'd_threshold_2': 82, 'd_threshold_3': 95, 'c_threshold_1': 3, 'c_threshold_2': 91, 'c_threshold_3': 91}


Best trial: 0. Best value: 2.70396:   1%|          | 2/300 [01:02<2:31:07, 30.43s/it]

[I 2026-11-21 08:42:21,749] Trial 1 finished with value: 2.5975714285714284 and parameters: {'d_threshold_1': 43, 'd_threshold_2': 82, 'd_threshold_3': 95, 'c_threshold_1': 3, 'c_threshold_2': 91, 'c_threshold_3': 91}. Best is trial 0 with value: 2.703964285714286.
Evaluating solution 0: {'d_threshold_1': 48, 'd_threshold_2': 57, 'd_threshold_3': 58, 'c_threshold_1': 60, 'c_threshold_2': 86, 'c_threshold_3': 88}


... (592 outputs left out)

Best trial: 37. Best value: 2.765: 100%|█████████▉| 299/300 [2:13:53<00:25, 25.96s/it]

[I 2026-11-21 10:55:13,288] Trial 298 finished with value: 2.6823928571428572 and parameters: {'d_threshold_1': 71, 'd_threshold_2': 89, 'd_threshold_3': 97, 'c_threshold_1': 33, 'c_threshold_2': 60, 'c_threshold_3': 70}. Best is trial 37 with value: 2.7649999999999997.
Evaluating solution 0: {'d_threshold_1': 74, 'd_threshold_2': 90, 'd_threshold_3': 97, 'c_threshold_1': 36, 'c_threshold_2': 56, 'c_threshold_3': 71}


Best trial: 37. Best value: 2.765: 100%|██████████| 300/300 [2:14:18<00:00, 26.86s/it]

[I 2026-11-21 10:55:37,687] Trial 299 finished with value: 2.7250357142857142 and parameters: {'d_threshold_1': 74, 'd_threshold_2': 90, 'd_threshold_3': 97, 'c_threshold_1': 36, 'c_threshold_2': 56, 'c_threshold_3': 71}. Best is trial 37 with value: 2.7649999999999997.


In [7]:
print("\n=== OPTIMIZATION COMPLETE ===")
print(f"Best score:  {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

importance = optuna.importance.get_param_importances(study)
print("\n=== PARAMETER IMPORTANCE ===")
for param, imp in importance.items():
    print(f"  {param}: {imp:.4f}")


=== OPTIMIZATION COMPLETE ===
Best score:  2.7650
Best params: {'d_threshold_1': 90, 'd_threshold_2': 96, 'd_threshold_3': 98, 'c_threshold_1': 57, 'c_threshold_2': 81, 'c_threshold_3': 94}

=== PARAMETER IMPORTANCE ===
  d_threshold_1: 0.6033
  c_threshold_1: 0.3967
